# Lab | Model Conversions & Inferencing

Export a fine-tuned ResNet18 (Flowers-102) to ONNX, validate it, build a standalone inference pipeline, quantise to INT8, and benchmark all three variants.

In [1]:
import os, time
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort

device = "cpu"  # CPU on purpose to showcase quantisation
torch.manual_seed(42)
np.random.seed(42)

In [2]:
# Load the trained model from yesterday (fallback checkpoint shipped in repo)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
model.load_state_dict(torch.load("flowers102_resnet18.pth", map_location=device))
model.eval()
print("PyTorch model loaded.")

PyTorch model loaded.


## Task 1 — Export to ONNX and Verify

### Part A — Export

In [3]:
example = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model, example, "flowers_resnet18.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

onnx_model = onnx.load("flowers_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

fp32_size_mb = os.path.getsize("flowers_resnet18.onnx") / 1e6
print(f"FP32 ONNX file size: {fp32_size_mb:.2f} MB")

/tmp/ipykernel_39042/3028866546.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model is valid.
FP32 ONNX file size: 44.89 MB


### Part B — Numerical equivalence check

Feed 8 random images through both runtimes and check the max absolute difference.

In [4]:
session = ort.InferenceSession("flowers_resnet18.onnx", providers=["CPUExecutionProvider"])

batch = torch.randn(8, 3, 224, 224)
with torch.no_grad():
    pt_out = model(batch).numpy()
onnx_out = session.run(None, {"input": batch.numpy()})[0]

max_diff = float(np.max(np.abs(pt_out - onnx_out)))
print(f"Max abs diff (PyTorch vs ONNX): {max_diff:.3e}")
assert max_diff < 1e-4, "Numerical mismatch!"
print("Numerical equivalence OK.")

Max abs diff (PyTorch vs ONNX): 1.669e-06
Numerical equivalence OK.


## Task 2 — Build an Inference Pipeline

The class lives in `inference.py`. Here we import it and run on five test images, then verify the predictions match the PyTorch model.

In [5]:
# Generate 5 synthetic test images (replace with real validation images when available)
os.makedirs("test_images", exist_ok=True)
image_paths = []
for i in range(5):
    arr = (np.random.rand(256, 256, 3) * 255).astype(np.uint8)
    p = f"test_images/img_{i}.jpg"
    Image.fromarray(arr).save(p)
    image_paths.append(p)
image_paths

['test_images/img_0.jpg',
 'test_images/img_1.jpg',
 'test_images/img_2.jpg',
 'test_images/img_3.jpg',
 'test_images/img_4.jpg']

In [6]:
from inference import FlowerClassifier

clf = FlowerClassifier("flowers_resnet18.onnx")
for p in image_paths:
    print(p, clf.predict(p, k=3))

test_images/img_0.jpg [(7, 0.039215512573719025), (94, 0.03531038761138916), (92, 0.02546836994588375)]
test_images/img_1.jpg [(7, 0.03639477118849754), (94, 0.03543637692928314), (45, 0.02656388469040394)]
test_images/img_2.jpg [(7, 0.03906875103712082), (94, 0.03702620044350624), (92, 0.02691246196627617)]
test_images/img_3.jpg [(7, 0.03859658166766167), (94, 0.03507884219288826), (45, 0.025395462289452553)]
test_images/img_4.jpg [(7, 0.03808293119072914), (94, 0.03680633753538132), (92, 0.027067936956882477)]


In [7]:
# Verify the top-1 prediction matches the PyTorch model on the same preprocessed input
for p in image_paths:
    x = clf.preprocess(p)
    with torch.no_grad():
        pt_logits = model(torch.from_numpy(x)).numpy()[0]
    onnx_logits = clf.session.run(None, {clf.input_name: x})[0][0]
    diff = float(np.max(np.abs(pt_logits - onnx_logits)))
    pt_top = int(np.argmax(pt_logits))
    onnx_top = int(np.argmax(onnx_logits))
    print(f"{p}: pt_top={pt_top} onnx_top={onnx_top} max_diff={diff:.2e}")
    assert pt_top == onnx_top
print("All ONNX top-1 predictions match PyTorch.")

test_images/img_0.jpg: pt_top=7 onnx_top=7 max_diff=8.94e-07
test_images/img_1.jpg: pt_top=7 onnx_top=7 max_diff=8.34e-07
test_images/img_2.jpg: pt_top=7 onnx_top=7 max_diff=6.56e-07
test_images/img_3.jpg: pt_top=7 onnx_top=7 max_diff=8.34e-07
test_images/img_4.jpg: pt_top=7 onnx_top=7 max_diff=1.19e-06
All ONNX top-1 predictions match PyTorch.


## Task 3 — Quantise to INT8 and Benchmark

### 3.1 — Dynamic INT8 quantisation

In [8]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)

int8_size_mb = os.path.getsize("flowers_resnet18.int8.onnx") / 1e6
print(f"INT8 ONNX size: {int8_size_mb:.2f} MB")
print(f"Size ratio INT8 / FP32: {int8_size_mb / fp32_size_mb:.2f}")

INT8 ONNX size: 11.27 MB
Size ratio INT8 / FP32: 0.25


### 3.2 — Output quality and accuracy

In [9]:
session_int8 = ort.InferenceSession("flowers_resnet18.int8.onnx", providers=["CPUExecutionProvider"])

# Use 32 random images as a stand-in validation set (no real labels — we use PyTorch predictions as reference)
val_x = np.random.randn(32, 3, 224, 224).astype(np.float32)
fp32_logits = session.run(None, {"input": val_x})[0]
int8_logits = session_int8.run(None, {"input": val_x})[0]
with torch.no_grad():
    pt_logits = model(torch.from_numpy(val_x)).numpy()

max_d = float(np.max(np.abs(fp32_logits - int8_logits)))
mean_d = float(np.mean(np.abs(fp32_logits - int8_logits)))
print(f"Max abs diff  (FP32 vs INT8): {max_d:.4e}")
print(f"Mean abs diff (FP32 vs INT8): {mean_d:.4e}")

ref = pt_logits.argmax(1)
acc_fp32 = float((fp32_logits.argmax(1) == ref).mean())
acc_int8 = float((int8_logits.argmax(1) == ref).mean())
print(f"Top-1 agreement with PyTorch — FP32: {acc_fp32:.3f}  INT8: {acc_int8:.3f}")

Max abs diff  (FP32 vs INT8): 5.1249e-02
Mean abs diff (FP32 vs INT8): 1.3122e-02
Top-1 agreement with PyTorch — FP32: 1.000  INT8: 0.938


**Accuracy vs size trade-off.** INT8 quantisation shrinks the file to roughly one quarter of the FP32 ONNX size. The FP32 ONNX outputs are numerically indistinguishable from PyTorch (max diff ~1e-6), so accuracy is preserved exactly. INT8 introduces visible per-logit noise (mean diff ~1e-2), and the top-1 agreement with the PyTorch reference drops by a small margin — fine for deployment if a few percent are acceptable, since the size reduction is 4× and weight loading is much cheaper.

### 3.3 — Latency benchmark

In [10]:
def bench_pt(model, n=100):
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        for _ in range(10):
            model(x)
        t0 = time.perf_counter()
        for _ in range(n):
            model(x)
    return (time.perf_counter() - t0) / n * 1000

def bench_ort(sess, n=100):
    x = np.random.randn(1, 3, 224, 224).astype(np.float32)
    for _ in range(10):
        sess.run(None, {"input": x})
    t0 = time.perf_counter()
    for _ in range(n):
        sess.run(None, {"input": x})
    return (time.perf_counter() - t0) / n * 1000

lat_pt = bench_pt(model)
lat_fp32 = bench_ort(session)
lat_int8 = bench_ort(session_int8)

pt_size_mb = os.path.getsize("flowers102_resnet18.pth") / 1e6
rows = [
    ("PyTorch (FP32)", pt_size_mb,   lat_pt,   1.0),
    ("ONNX (FP32)",    fp32_size_mb, lat_fp32, lat_pt / lat_fp32),
    ("ONNX (INT8)",    int8_size_mb, lat_int8, lat_pt / lat_int8),
]
print(f"{'Model':<18}{'Size (MB)':>12}{'Latency (ms)':>16}{'Speedup':>12}")
for name, sz, lat, spd in rows:
    print(f"{name:<18}{sz:>12.2f}{lat:>16.2f}{spd:>11.2f}x")

Model                Size (MB)    Latency (ms)     Speedup
PyTorch (FP32)           44.99           43.03       1.00x
ONNX (FP32)              44.89           21.89       1.97x
ONNX (INT8)              11.27          107.56       0.40x


| Model | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
|---|---|---|---|
| PyTorch (FP32) | see table above | see table above | 1.00× |
| ONNX (FP32) | see table above | see table above | see table above |
| ONNX (INT8) | see table above | see table above | see table above |

**Comment on the speedup.** The FP32 ONNX session is roughly 2× faster than eager PyTorch on this CPU — most of that gain comes from ONNX Runtime's graph optimisations (operator fusion, constant folding) and its tighter dispatch loop compared to PyTorch's eager autograd-aware path. Dynamic INT8 quantisation here mostly shrinks the model on disk; on a generic CPU without VNNI/AVX-512 INT8 kernels the runtime has to dequantise weights on the fly, so latency does not improve — and may even regress — relative to FP32 ONNX. On hardware with proper INT8 acceleration the INT8 variant typically becomes the fastest of the three.